# 1.0 - Bibliotecas

In [23]:
import pandas as pd
import kagglehub
import warnings
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# 2.0 - Carregando dados

In [24]:
data = kagglehub.dataset_download("sujalsuthar/amazon-delivery-dataset")

print("Path to dataset files:", data)

Path to dataset files: /home/thomas-linux/.cache/kagglehub/datasets/sujalsuthar/amazon-delivery-dataset/versions/1


In [25]:
data = pd.read_csv(data + "/amazon_delivery.csv")
data

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43734,jlxf819993117,30,4.8,26.902328,75.794257,26.912328,75.804257,2022-03-24,11:35:00,11:45:00,Windy,High,motorcycle,Metropolitian,160,Home
43735,aevx342135787,21,4.6,0.000000,0.000000,0.070000,0.070000,2022-02-16,19:55:00,20:10:00,Windy,Jam,motorcycle,Metropolitian,180,Jewelry
43736,xnek760674819,30,4.9,13.022394,80.242439,13.052394,80.272439,2022-03-11,23:50:00,00:05:00,Cloudy,Low,scooter,Metropolitian,80,Home
43737,cynl434665991,20,4.7,11.001753,76.986241,11.041753,77.026241,2022-03-07,13:35:00,13:40:00,Cloudy,High,motorcycle,Metropolitian,130,Kitchen


# 3.0 - Limpeza dos dados

In [26]:
limpeza1 = data.copy()

In [7]:
limpeza1.shape

(43739, 16)

In [8]:
limpeza1.dtypes

Order_ID            object
Agent_Age            int64
Agent_Rating       float64
Store_Latitude     float64
Store_Longitude    float64
Drop_Latitude      float64
Drop_Longitude     float64
Order_Date          object
Order_Time          object
Pickup_Time         object
Weather             object
Traffic             object
Vehicle             object
Area                object
Delivery_Time        int64
Category            object
dtype: object

In [9]:
limpeza1.isnull().sum()

Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather            91
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64

In [10]:
# Por uma questão de simplicidade, vamos remover as linhas que contêm valores de clima nulos. Porém, outras técnicas podem ser abordaddas para lidar com valores nulos, como a substituição pela média, mediana ou moda.
# A classificação do agente iremos tratar mais a frente utilizando a média dos valores.
limpeza2 = limpeza1.dropna(subset=['Weather'])

In [11]:
limpeza2.isnull().sum()

Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather             0
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64

# 4.0 - Preprocessamento e Algoritmos

In [12]:
X = limpeza2.drop(columns=['Delivery_Time'], axis=1)
y = limpeza2['Delivery_Time']

In [13]:
categorical_columns = X.select_dtypes(include=['object', 'category']).columns
numeric_columns = X.select_dtypes(include=['number']).columns

print("Colunas categóricas:", categorical_columns)
print("Colunas numéricas:", numeric_columns)

Colunas categóricas: Index(['Order_ID', 'Order_Date', 'Order_Time', 'Pickup_Time', 'Weather',
       'Traffic', 'Vehicle', 'Area', 'Category'],
      dtype='object')
Colunas numéricas: Index(['Agent_Age', 'Agent_Rating', 'Store_Latitude', 'Store_Longitude',
       'Drop_Latitude', 'Drop_Longitude'],
      dtype='object')


In [14]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categoric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_columns),
        ('cat', categoric_transformer, categorical_columns)
    ]
)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [16]:
warnings.filterwarnings('ignore')
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'ElasticNet Regression': ElasticNet(),
    'Support Vector Regression': SVR(),
    'Random Forest': RandomForestRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(),
    'LightGBM': LGBMRegressor()
}

In [17]:
results = []

for name, model in models.items():
    try:
        pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('model', model)])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        results.append({'Model': name, 'Mean Absolute Error': mae, 'R2 Score': r2})
    except Exception as e:
        results.append({'Model': name, 'Error': str(e)})

results_df = pd.DataFrame(results)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000290 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1942
[LightGBM] [Info] Number of data points in the train set: 34918, number of used features: 444
[LightGBM] [Info] Start training from score 124.701644


In [18]:
results_df_cleaned = results_df[['Model', 'Mean Absolute Error', 'R2 Score']]

results_df_cleaned = results_df_cleaned.dropna()

results_df_cleaned = results_df_cleaned.sort_values(by='Mean Absolute Error')

results_df_cleaned

,Model,Mean Absolute Error,R2 Score
7,LightGBM,18.906620,0.783491
5,Random Forest,19.421389,0.754310
6,Gradient Boosting,22.008803,0.697746
1,Ridge Regression,25.326413,0.625615
0,Linear Regression,25.329750,0.625539
4,Support Vector Regression,25.859049,0.585853
2,Lasso Regression,27.447345,0.543930
3,ElasticNet Regression,34.522367,0.273231


# 6.0 - GridSearch

In [22]:
param_grids = {
    'Random Forest': {'model__n_estimators': [100], 'model__max_depth': [10, 20]},
    'LightGBM': {'model__n_estimators': [100], 'model__learning_rate': [0.01]}
}

results_randomizedSearch = []

for name, model in models.items():
    if name in param_grids:
        pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('model', model)])
        random_search = RandomizedSearchCV(pipeline, param_distributions=param_grids[name], 
                                           n_iter=10, cv=3, scoring='neg_mean_absolute_error', random_state=42)
        random_search.fit(X_train, y_train)
        best_model = random_search.best_estimator_
        y_pred = best_model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        results_randomizedSearch.append({'Model': name, 'Best Params': random_search.best_params_, 
                                   'Mean Absolute Error': mae, 'R2 Score': r2})

results_randomizedSearch_df = pd.DataFrame(results_randomizedSearch)

print("Resultados com GridSearch:")
results_randomizedSearch_df

KeyboardInterrupt: 

# 7.0 - Avaliação final

In [ ]:
pipeline = Pipeline(
    steps=[('preprocessor', preprocessor),
           ('model', LinearRegression()),]
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
y_pred_df = pd.DataFrame(y_pred, columns=['Delivery_Time_predicted'])

# 8.0 - Tranformação em produto

In [62]:
y_test_reset = y_test.reset_index(drop=True)

y_pred_df = pd.DataFrame(y_pred, columns=['Delivery_Time_predicted'])

X_test_reset = X_test.reset_index(drop=True)

result_df = pd.concat([X_test_reset, y_test_reset, y_pred_df], axis=1)

result_df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Category,Delivery_Time,Delivery_Time_predicted
0,qbxi696425797,22,4.2,26.902940,75.793007,26.962940,75.853007,2022-03-13,19:15:00,19:30:00,Cloudy,Jam,motorcycle,Metropolitian,Kitchen,170,156.548634
1,mmki589631517,31,4.6,22.753659,75.903365,22.863659,76.013365,2022-03-29,18:20:00,18:30:00,Fog,Medium,motorcycle,Urban,Outdoors,180,160.357225
2,vipx590722806,28,4.8,12.323978,76.627961,12.343978,76.647961,2022-03-13,11:45:00,11:50:00,Cloudy,High,scooter,Metropolitian,Snacks,100,138.141106
3,jlet363874990,29,4.6,12.325461,76.632278,12.415461,76.722278,2022-03-06,18:55:00,19:10:00,Cloudy,Medium,motorcycle,Metropolitian,Clothing,170,170.993008
4,ztbf292297028,25,4.7,19.120083,72.907385,19.170083,72.957385,2022-03-13,19:55:00,20:10:00,Sunny,Jam,scooter,Metropolitian,Toys,75,99.703562
